---
# ResNet Implementation
---


---
## Google Colab
---

Training a ResNet on CIFAR-10 is much faster with a GPU and <span style="color:#E74C3C">highly recommended</span>. If your local machine has no CUDA-capable GPU, use Google Colab.

1. Open this notebook in Colab.
2. Upload the full exercise folder, including `shortcuts.py`, `blocks.py`, `resnet.py`, `training.py`, `data.py`, `tests.py`, and `visual.py`.
3. Select **Runtime -> Change runtime type -> Hardware accelerator -> GPU**.
4. Run the notebook cells as usual.

If you use Colab from VS Code, make sure you edit the files on the Colab runtime or synchronize your local changes before running tests.

---
# Imports
---

Run the code cell below after you have set up the exercise files correctly.

In [33]:
import importlib
import shortcuts
import blocks
import data
import resnet
import training
import tests
import visual

---
## CIFAR-10 Dataset
---

The data loading and transformations are provided in `data.py`.

For training, the provided loader uses:

- random crop with padding,
- random horizontal flip,
- conversion to floating point tensors,
- CIFAR-10 normalization.

The test loader uses only conversion and normalization. You do not need to implement data loading.

In [34]:
visual.cifar10_pictures().show()

---
## Residual Network Design
---

Implement the CIFAR version of ResNet.

Useful PyTorch modules and functions are the following. Look them up.

- `nn.Conv2d`
- `nn.BatchNorm2d`
- `nn.ReLU`
- `nn.Identity`
- `nn.Sequential`
- `nn.MaxPool2d`
- `nn.AdaptiveAvgPool2d`
- `nn.Linear`
- `torch.flatten`
- `torch.argmax`
- `torch.cat`
- `torch.zeros`, `tensor.new_zeros`

---
## **Task:** Shortcuts
---

Implement the shortcut layers in `shortcuts.py`:

- `ZeroPadShortcut (Option A)`, the parameter-free shortcut.
- `ProjectionShortcut (Option B)`, a learned shortcut using a $1 \times 1$ convolution.

Hints:

- For zero-padding, first downsample spatially if `stride > 1`.
- Use `nn.MaxPool2d(...)` for the zero-padding shortcut's spatial downsampling.
- Use `torch.zeros` or `x.new_zeros` for channel padding.
- Use `torch.cat` to append zero channels.
- The learned projection should use `nn.Conv2d(...)`.
- You may include `nn.BatchNorm2d` in the projection shortcut.

In [35]:
importlib.reload(shortcuts)
importlib.reload(resnet)
tests.test_zero_pad_shortcut_module()
tests.test_projection_module()

PASS: ZeroPadShortcut is correct.
PASS: ProjectionShortcut is correct.


True

---
## **Task:** BasicBlock and ResNet
---

Implement the model components in `blocks.py` and `resnet.py`:

- `BasicBlock`,
- `make_block_group`,
- `ResNet`

Your implementation should support both shortcut styles:

- `shortcut="zero_pad"`,
- `shortcut="projection"`.

Hints:

- Use `padding=1` with $3 \times 3$ convolutions to preserve spatial size when stride is 1.
- Add the shortcut branch to the residual branch before the final activation.
- Use `nn.Identity()` when the shortcut does not need to change shape.
- A block group represents one part of the network that produces one feature-map size and one channel count (see ResNet paper, Figure 3 coloring)
- `make_block_group` should return an `nn.Sequential` of residual blocks.
- A small model such as `num_blocks=(1, 1, 1)` is useful for debugging before trying ResNet-20.

In [36]:
importlib.reload(blocks)
importlib.reload(resnet)
tests.test_basic_block()
tests.test_resnet_model()

PASS: BasicBlock is correct.
PASS: ResNet model is correct.


True

---
## **Task:** Training and Learning Rate Schedule
---

Implement the training utilities in `training.py`:

- `step_learning_rate`,
- `initialize`,
- `training_step`,
- `train_one_epoch`,
- `evaluate_accuracy`,
- the epoch loop inside `train`.

The function `set_learning_rate` is already implemented for you. Your scheduler should only compute the learning rate value. Use `set_learning_rate` to write it into the optimizer.

Hints:

- The paper uses SGD with momentum.
- Use `nn.CrossEntropyLoss` for classification.
- Move the model, inputs and labels to the selected device with `.to(device)`.
- Because of BatchNorm: Use `model.train()` during training and `model.eval()` during evaluation.
- Use `torch.no_grad()` during evaluation.
- Return Python floats for losses and accuracies.
- Use `tensor.item()` to transfer (loss) values from GPU to CPU. 
- Don't forget to apply learning rate decay.

In [37]:
importlib.reload(resnet)
importlib.reload(training)
importlib.reload(resnet)
tests.test_learning_rate_schedule()
tests.test_training_step_and_epoch()
tests.test_evaluate_accuracy()

PASS: learning-rate schedule is correct.
PASS: training_step and train_one_epoch are correct.
PASS: evaluate_accuracy is correct.


True

---
## Train the Model
---

Train your ResNet on CIFAR-10. Aim for about **87-89% test accuracy** with a shorter run (e.g. 25 epochs).

<span style="color:#E74C3C">Note that one epoch can take 30 seconds even with a GPU.</span>

The paper hyperparameters are good, but feel free to tune:

- number of epochs,
- batch size,
- learning rate,
- learning-rate milestones,
- zero-padding vs. projection shortcuts,
- number of blocks per block group,
- channel counts.

Start with ResNet-20: `num_blocks=(3, 3, 3)` and channels `(16, 32, 64)`. If training is slow, first verify the implementation with a smaller model.

<span style="color:#E74C3C">Correctness:</span> According to Figure 6 in the ResNet paper, we need to reach $>90\%$ test accuracy to outperform shallow networks with 20-layers. 

The solution (ResNet-20: `num_blocks=(3, 3, 3)` and channels `(16, 32, 64)`) reaches $90.81\%$ test accuracy after 45 epochs.

In [38]:
##################################################
# TODO
import importlib
importlib.reload(training)
importlib.reload(resnet)

model, train_loss, test_acc = resnet.train(
    lr=.1,
    batch_size=128,
    momentum=0.0001,
    weight_decay=0.00008,
    epochs=40,
    milestones=(20,30),
    shortcut="projection",
)

##################################################

visual.show_training_stats(train_loss, test_acc).show()

Using device: cuda


LR: 0.0010, Train loss: 0.491, Test accuracy: 80.06 %: 100%|██████████| 40/40 [23:25<00:00, 35.15s/it]


In [39]:
visual.cifar10_predictions(model).show()